In [27]:
# Load libs
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import altair as alt

In [ ]:
# Load People.csv, FieldingPost.csv, PitchingPost.csv, BattingPost.csv, Schools.csv, Salaries.csv, etc.
DATA_PATH = "baseball/core/"
ppl  = pd.read_csv(DATA_PATH + "People.csv")
fld  = pd.read_csv(DATA_PATH + "FieldingPost.csv")
pch  = pd.read_csv(DATA_PATH + "PitchingPost.csv")
bat  = pd.read_csv(DATA_PATH + "BattingPost.csv")
scl  = pd.read_csv(DATA_PATH + "Schools.csv")
sal  = pd.read_csv(DATA_PATH + "Salaries.csv")
col  = pd.read_csv(DATA_PATH + "CollegePlaying.csv")
infCoef = pd.read_csv("baseball/infCoef.csv")

In [29]:
# "Clean" data
fld.drop(['CS','SB','PB'], axis="columns", inplace=True)
ppl.drop(['birthMonth','birthDay','birthCountry',
       'birthState','birthCity','deathYear','deathMonth','deathDay',
       'deathCountry','deathState','deathCity'], axis="columns", inplace=True)
ppl['finalGame'] = ppl.finalGame.astype('datetime64[ns]')
ppl = ppl[ppl.finalGame.dt.year > 1900] # modern baseball rules established in 1901
sal = sal[sal.yearID > 1900]

# Join data
college = col.merge(scl, on='schoolID', how='left').iloc[:, :-1]

# Apply coef on everyones salary
sal = sal.merge(infCoef, on="yearID", how="left")
sal["finalSalary"] = sal.coefficient * sal.salary

In [9]:
last_college = college.groupby('playerID').max()
last_college.reset_index(inplace=True)
last_college = last_college[last_college.yearID > 1984]

zero MLB players attended brown cornell john hopkins chicago or tufts. academic prestige (without a baseball program to match) doesnt feed the majors

In [ ]:
sorta_ivy = ['Brown University', 'Columbia University', 'Cornell University',
             'Dartmouth College', 'Harvard University', 'University of Pennsylvania',
             'Princeton University', 'Yale University', 'Northwestern University',
             'University of Notre Dame', 'Georgetown University',
             'Johns Hopkins University', 'University of Chicago', 'Tufts University',
             'Massachusetts Institute of Technology']  

sporty = ['University of Southern California', 'Louisiana State University', 'Texas A&M University',
          'University of Texas at Austin', 'University of Arizona', 'Arizona State University',
          'University of Oklahoma', 'Oklahoma State University', 'University of Miami', 'Florida State University',
          'University of Florida', 'University of Georgia', 'Mississippi State University', 'University of Mississippi',
          'Auburn University', 'University of South Carolina', 'Clemson University',
          'University of California, Los Angeles',
          'California State University Fullerton', 'California State University Long Beach',
          'California State University Fresno',
          'Wichita State University', 'Pepperdine University', 'Baylor University',
          'University of North Carolina at Chapel Hill', 'Oregon State University', 'Coastal Carolina University',
          'Stanford University', 'Vanderbilt University', 'Duke University', 'Rice University']

last_college = last_college.assign(
    school_type = lambda x: np.select(
        [x.name_full.isin(sorta_ivy), x.name_full.isin(sporty)],
        [1, 2],
        default=0
    )
)

,playerID,career_total,peak_salary,avg_salary,years_played
0,aardsda01,10888994.95,5176800.0,1.555571e+06,7
1,aasedo01,5152505.00,1477035.0,1.288126e+06,4
2,abadan01,419802.60,419802.6,4.198026e+05,1
3,abadfe01,4137729.08,1347875.0,8.275458e+05,5
4,abbotje01,1495701.50,438480.0,3.739254e+05,4
...,...,...,...,...,...
5144,zumayjo01,5019476.10,1610560.0,8.365794e+05,6
5145,zuninmi01,1122639.42,571557.3,5.613197e+05,2
5146,zupcibo01,788740.80,397713.0,2.629136e+05,3
5147,zuvelpa01,302731.00,302731.0,3.027310e+05,1


Problems:
- salary reflects skill
- Small ivy size

In [ ]:
player_salary = sal.groupby("playerID").agg(
    career_total = ("finalSalary", "sum"),
    peak_salary  = ("finalSalary", "max"),
    avg_salary   = ("finalSalary", "mean"),
    years_played = ("finalSalary", "count")
).reset_index()

analysis_df = player_salary.merge(
    last_college[["playerID", "school_type"]],
    on="playerID", how="inner"
)

label_map = {0: "Others", 1: "Academics", 2: "Sport Schools"}
analysis_df["school_type_label"] = analysis_df["school_type"].map(label_map)

alt.Chart(...)

In [111]:
avg_chart = alt.Chart(analysis_df).mark_boxplot().encode(
    x=alt.X("school_type_label:N", title="School Type", sort=["Others", "Academics", "Sport Schools"]),
    y=alt.Y(
        "avg_salary:Q",
        scale=alt.Scale(type="log"),
        axis=alt.Axis(format="$,.0f", title="Career Average Salary ($, log scale)")
    )
).properties(
    width=200,
    title='How Well Were Players Paid Across Career?'
)

peak_chart = alt.Chart(analysis_df).mark_boxplot().encode(
    x=alt.X("school_type_label:N", title="School Type", sort=["Others", "Academics", "Sport Schools"]),
    y=alt.Y(
        "peak_salary:Q",
        scale=alt.Scale(type="log"),
        axis=alt.Axis(format="$,.0f", title="Career Peak Salary ($, log scale)")
    )
).properties(
    width=200,
    title='How Well Were Players Paid At Their Best?'
)

years_chart = alt.Chart(analysis_df).mark_boxplot().encode(
    x=alt.X("school_type_label:N", title="School Type", sort=["Others", "Academics", "Sport Schools"]),
    y=alt.Y(
        "years_played:Q",
        axis=alt.Axis(title="Career Years Played")
    )
).properties(
    width=200,
    title='How Long Did Players Play?'
)

combined_chart = avg_chart | peak_chart | years_chart
combined_chart

alt.HConcatChart(...)

In [ ]:
# Look at salary over time 
stat_sal = pd.DataFrame()

# Take the summary stats
stat_sal['Mean']          = sal.groupby("yearID")["finalSalary"].mean()
stat_sal['Median']        = sal.groupby("yearID")["finalSalary"].median()
stat_sal['.25 Quartile']  = sal.groupby("yearID")["finalSalary"].quantile(0.25)
stat_sal['.75 Quartile']  = sal.groupby("yearID")["finalSalary"].quantile(0.75)
stat_sal.reset_index(inplace=True)
stat_sal = stat_sal.melt(id_vars="yearID")

# NOTE: Salary only starts at 1985 (womp-womp)

# Plot trend
sal_chart = alt.Chart(stat_sal).mark_line(interpolate="monotone").encode(
    x=alt.X(
        "yearID:Q",
        title='Year',
        axis=alt.Axis(labelAngle=-45, format='d'),
        scale=alt.Scale(domain=[1985, stat_sal.yearID.max()])
    ),
    y=alt.Y(
        "value:Q",
        title='Salary ($)',
        axis=alt.Axis(format='$,.0f')
        ),
    color=alt.Color(
    "variable:N",
    sort=["Mean", "Median", ".25 Quartile", ".75 Quartile"],
    legend=alt.Legend(title='Summary Stat.')
    )
).properties(
    title={
        'text':'MLB Players Salary Trends',
        'subtitle':'Salaries are rising'
    }
)

sal_chart = sal_chart.configure_title(
    fontSize=18,
    anchor='start'
)

In [ ]:
'''
Talking points:
- mean higher than 75th percentile
- All have a positive increase, even 25 percentile
'''
sal_chart.show()

alt.Chart(...)